[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/metaflow-certified/notebooks/day-01-intro-metaflow.ipynb#scrollTo=a1b2c3d4)

---
# Day 1 · Introduction to Metaflow: Why Workflows Matter
**certified-journeys / metaflow-certified** · Day 1 · Learn

> **Goal for today:** Install Metaflow, understand why workflow orchestration matters for ML, write your first `HelloWorldFlow`, run it locally, and inspect the execution output.

In [ ]:
%pip install -q metaflow

## Step 1 · Why Workflow Orchestration?

Before Metaflow, ML engineers ran notebooks or ad-hoc scripts that were:
- **Hard to reproduce** — environment, data, and code state were implicit
- **Hard to scale** — moving from laptop to cloud required rewriting
- **Hard to debug** — no checkpointing meant re-running everything on failure

Metaflow solves all three by making **every step a checkpoint** with versioned data artifacts.

| Without Metaflow | With Metaflow |
|---|---|
| Script runs top-to-bottom, no checkpoints | Each `@step` is independently rerunnable |
| Data passed via files or globals | Data versioned as `self.attribute` artifacts |
| Scaling = rewrite | Same code runs local or on AWS Batch/K8s |
| Debugging = start over | `resume` from failed step |

Metaflow was originally built at Netflix for production ML — it's designed to be **simple by default, powerful when you need it**.

In [ ]:
# Let's verify Metaflow installed correctly and check the version
import metaflow
print(f"Metaflow version: {metaflow.__version__}")

# Metaflow ships with a CLI — let's confirm it's accessible
import subprocess
result = subprocess.run(["python", "-m", "metaflow", "--version"],
                        capture_output=True, text=True)
print(result.stdout.strip() or result.stderr.strip())
print("\nMetaflow is ready to use!")

**What just happened?**

- We imported `metaflow` and confirmed it's installed correctly.
- **Metaflow has both a Python API and a CLI** — you use Python to define flows and the CLI (or `python flow.py run`) to execute them.
- The version output confirms we have a working environment.
- In Colab, `%pip install` adds the package to the current kernel session.

## Step 2 · Anatomy of a Metaflow Flow

Every Metaflow flow is a Python class that:
1. **Inherits from `FlowSpec`** — the base class that turns a Python class into a DAG
2. **Decorates methods with `@step`** — each decorated method is a node in the graph
3. **Calls `self.next()`** — explicit transition tells Metaflow what comes next
4. **Starts with `start` and ends with `end`** — every flow must have exactly these two step names

```
start  →  [your steps]  →  end
```

The `%%writefile` magic below writes a `.py` file to disk so we can run it with `!python`.
This is the correct way to work with Metaflow in Jupyter/Colab — flows must be Python scripts.

In [ ]:
%%writefile hello_flow.py
from metaflow import FlowSpec, step

class HelloWorldFlow(FlowSpec):
    """
    A minimal Metaflow flow that prints a greeting.
    This is the canonical starting point for every Metaflow journey.
    """

    @step
    def start(self):
        # The start step is always named 'start' — it's the entry point
        print("Starting the HelloWorldFlow!")
        self.greeting = "Hello from Metaflow"
        # self.next() explicitly declares the next step in the DAG
        self.next(self.greet)

    @step
    def greet(self):
        # self.greeting was set in the previous step — Metaflow serialized it automatically
        print(f"{self.greeting}! 🚀")
        print("Each step is checkpointed. If this step fails, we resume here.")
        self.next(self.end)

    @step
    def end(self):
        # The end step is always named 'end' — it's the exit point
        print("Flow complete! All steps ran successfully.")

if __name__ == '__main__':
    HelloWorldFlow()

**What just happened?**

- `%%writefile` saved the code to `hello_flow.py` on disk — this is **not** running yet.
- **`FlowSpec`** is the base class — it introspects the `@step` methods to build the DAG.
- **`self.next(self.greet)`** is explicit wiring — there is no magic ordering by method position.
- `self.greeting` set in `start` is automatically available in `greet` — **Metaflow serializes it between steps**.
- The `if __name__ == '__main__': FlowSpec()` pattern is required for Metaflow to work correctly.

## Step 3 · Running a Flow Locally

You run Metaflow flows from the command line:

```bash
python flow.py run              # run the flow
python flow.py show             # show the DAG structure
python flow.py run --help       # all run options
python flow.py run --max-workers 4  # parallel branches with 4 workers
```

In Colab/Jupyter, prefix with `!` to run shell commands.

The output format is:
```
[2024-01-01 12:00:00.000] FlowName/run_id/step_name/task_id (pid 12345)
```
Each line includes a **run ID** (auto-incrementing integer) and **task ID** that uniquely identifies this execution.

In [ ]:
# Run the flow — the ! prefix passes the command to the shell
# --no-pylint suppresses linting in Colab (optional in local dev)
!python hello_flow.py run --no-pylint 2>&1

**What just happened?**

- Metaflow executed `start → greet → end` in order, printing each step's output.
- **Every run gets a unique integer run ID** (you'll see it in the output like `HelloWorldFlow/1`).
- Metaflow created a `.metaflow/` directory in your working directory — this is the **local datastore** where artifacts and metadata are persisted.
- **`self.greeting` was serialized to disk** between steps, which is how Metaflow guarantees data durability and enables resume.
- The `[pid XXXXX]` in each line shows that steps run as separate OS processes — enabling true isolation.

## Step 4 · Inspecting Execution Logs and the Datastore

After running a flow, Metaflow stores everything in `.metaflow/`:

```
.metaflow/
  HelloWorldFlow/
    1/                    ← run ID
      start/
        0/                ← task ID
          data/           ← serialized artifacts (pickle)
          log_location    ← stdout/stderr logs
```

You can also use the **Metaflow Client API** to programmatically access runs without touching the filesystem directly.

In [ ]:
import os

# Show the .metaflow directory structure created by our run
metaflow_dir = ".metaflow"
if os.path.exists(metaflow_dir):
    print("Contents of .metaflow directory:")
    for root, dirs, files in os.walk(metaflow_dir):
        # Calculate depth for indentation
        depth = root.replace(metaflow_dir, "").count(os.sep)
        indent = "  " * depth
        print(f"{indent}{os.path.basename(root)}/")
        sub_indent = "  " * (depth + 1)
        for file in files:
            size = os.path.getsize(os.path.join(root, file))
            print(f"{sub_indent}{file}  ({size} bytes)")
else:
    print("No .metaflow directory found — did the flow run complete?")

In [ ]:
# Use the Metaflow Client API to inspect the most recent run
from metaflow import Flow, get_metadata

# get_metadata() tells us where Metaflow is storing data
print(f"Metaflow metadata provider: {get_metadata()}")
print()

# Flow() gives us access to all runs of HelloWorldFlow
flow = Flow("HelloWorldFlow")

# latest_run returns the most recently started run
run = flow.latest_run
print(f"Latest run ID: {run.id}")
print(f"Run successful: {run.successful}")
print(f"Run created at: {run.created_at}")
print()

# List all steps in this run
print("Steps in this run:")
for step in run:
    for task in step:
        # Each task has data — access artifacts by name
        data = task.data
        print(f"  Step: {step.id:10s} | Task: {task.id} | successful: {task.successful}")

**What just happened?**

- **`.metaflow/` is the local datastore** — it mirrors what would go to S3/GCS in production.
- `Flow("HelloWorldFlow")` is the **Client API** — the primary way to programmatically access past runs.
- `run.successful` is a boolean that tells you if every step completed without error.
- **Each run has a unique integer ID** — this is how Metaflow versions your experiments.
- `task.data` gives you access to all artifacts (like `self.greeting`) from that task.

## Step 5 · Accessing Artifacts from a Completed Run

One of Metaflow's most powerful features is **time-travel access to artifacts**.
Any `self.attribute` set in any step becomes a versioned artifact you can retrieve later.

This works because Metaflow **serializes every self-attribute** using Python's `pickle` (or custom serializers for NumPy, pandas, etc.) and stores it alongside the run metadata.

The Client API pattern is:
```python
from metaflow import Flow
run = Flow('MyFlow').latest_run          # or Flow('MyFlow')['42']
artifact = run['step_name'].task.data.attribute_name
```

In [ ]:
from metaflow import Flow

# Retrieve a specific artifact from our completed run
run = Flow("HelloWorldFlow").latest_run

# Access the 'greet' step
greet_step = run["greet"]

# Get the single task from that step
task = next(iter(greet_step))  # greet_step is iterable

# Retrieve the 'greeting' artifact — this is the self.greeting we set in 'start'
# (artifacts flow forward through steps automatically)
greeting_artifact = task.data.greeting
print(f"Retrieved artifact 'greeting': {greeting_artifact!r}")

# Show all artifact names available on this task
print("\nAll artifacts available on the 'greet' task:")
for name in dir(task.data):
    if not name.startswith("_"):
        print(f"  - {name}")

**What just happened?**

- We retrieved the `greeting` artifact **after the flow completed** — no need to keep the process alive.
- **Artifacts persist across Python sessions** — you can close the notebook and re-open it, and the artifacts are still there.
- `task.data` is a lazy-loading proxy — artifacts are only deserialized (unpickled) when you access them.
- **This is the foundation of reproducibility** — any run, any step, any artifact is accessible by run ID forever (until you delete it).

In [ ]:
# Challenge: Write a two-step Metaflow flow that:
# 1. In 'start': creates a list of 5 numbers and stores it as self.numbers
# 2. In a 'compute' step: calculates the sum and mean, stores them as self.total and self.mean
# 3. In 'end': prints the results
# Then:
#   a. Run the flow with !python
#   b. Use the Client API to retrieve self.total and self.mean from the completed run

# Scaffold — replace the ... with your code
%%writefile challenge_flow.py
from metaflow import FlowSpec, step

class NumberFlow(FlowSpec):

    @step
    def start(self):
        # TODO: create self.numbers = [1, 2, 3, 4, 5]
        ...
        self.next(self.compute)

    @step
    def compute(self):
        # TODO: compute self.total and self.mean from self.numbers
        ...
        self.next(self.end)

    @step
    def end(self):
        # TODO: print self.total and self.mean
        ...

if __name__ == '__main__':
    NumberFlow()

# Then run it:
# !python challenge_flow.py run --no-pylint

# Then retrieve the artifacts:
# from metaflow import Flow
# run = Flow('NumberFlow').latest_run
# ...

---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| `FlowSpec` | Base class for all Metaflow flows — turns a Python class into a DAG |
| `@step` | Decorator that marks a method as a DAG node (checkpoint) |
| `self.next()` | Explicit transition — you declare the DAG edges yourself |
| `start` / `end` | Every flow must begin with `start` and finish with `end` |
| `self.attribute` | Any attribute set on `self` in a step becomes a versioned artifact |
| `.metaflow/` | Local datastore — artifacts and metadata persist here between runs |
| Client API | `Flow('Name').latest_run` — programmatic access to past runs and artifacts |
| `python flow.py run` | Execute a flow; use `!` prefix in Colab/Jupyter |

> **Tip:** Metaflow's superpower is the seamless transition from local to cloud — always prototype locally first.

---
## What's next
**Day 2** → Flow Architecture and DAG Fundamentals — build a 4+ step linear flow, understand how data flows through self-attributes, and visualize your flow graph.

Mark Day 1 complete in your [tracker](../index.html).